# Copulas & Tail Dependence: Why Markets Crash Together More Often Than Correlation Says

A correlation matrix is a photograph of dependence taken in calm weather. The question risk managers actually get paid to answer — **if the S&P has one of its worst weeks, what happens to London and Tokyo?** — lives in the corner of the joint distribution, where a single number cannot see. Copulas separate what each market does on its own from how they move together, and once you make that split on 25 years of real index data, the verdict is blunt: the Gaussian copula says joint crashes are vanishingly rare; the data says they cluster; and a Student-t copula with 3.5 degrees of freedom repairs most of the damage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from scipy import stats
from scipy.special import gammaln
from statsmodels.distributions.empirical_distribution import ECDF

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
SEED = 11

## 1. Dependence is not correlation

The sample is 1,302 aligned weekly returns for the S&P 500, FTSE 100 and Nikkei 225, 2000-01-03 to 2024-12-31 — dot-com bust, 2008, the euro crisis, COVID and the 2022 rates shock all included. Before any statistics, two traps come free with global indices, and both are worth learning to spot. First, each exchange keeps its own holiday calendar, so returns are computed jointly and any week missing a market is dropped. Second — subtler — the markets do not even trade at the same time. Tokyo closes roughly ten hours before London and some fifteen before New York, so a same-calendar-day Nikkei close **leads** the US close, and daily cross-market correlations are structurally understated: the two prices are answering questions asked at different moments. Resampling to weekly (Friday-to-Friday) returns absorbs most of that offset — watch the SPX–Nikkei Pearson correlation jump from `0.15` on daily data to `0.58` on weekly.

Even then, “the correlation” is three different numbers. Pearson measures linear co-movement and is hostage to outliers; Spearman and Kendall are rank-based, so they see only the ordering — exactly the part a copula models. Kendall’s τ has the cleanest interpretation: the probability that two randomly chosen weeks agree in direction, minus the probability they disagree.

In [ ]:
px = yf.download(["^GSPC", "^FTSE", "^N225"], start="2000-01-01",
                 end="2025-01-01", auto_adjust=True, progress=False)["Close"]
px = px.rename(columns={"^GSPC": "SPX", "^FTSE": "FTSE", "^N225": "NKY"})[["SPX", "FTSE", "NKY"]]

wk  = px.resample("W-FRI").last()
ret = np.log(wk / wk.shift(1)).dropna()      # holiday alignment via dropna

ret_d = np.log(px / px.shift(1)).dropna()
print(f"{len(ret)} aligned weekly observations")
print(f"daily  Pearson SPX-NKY: {ret_d['SPX'].corr(ret_d['NKY']):.3f}   <- timezone artefact")
print(f"weekly Pearson SPX-NKY: {ret['SPX'].corr(ret['NKY']):.3f}")

In [ ]:
for m in ("pearson", "spearman", "kendall"):
    print(f"\n{m.capitalize()}")
    print(ret.corr(method=m).round(3))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(ret.corr(method="kendall"), annot=True, fmt=".2f",
            cmap="crest", vmin=0, vmax=1, square=True, ax=ax)
ax.set_title("Kendall's tau — weekly returns, 2000–2024");
plt.show()

## 2. Sklar's theorem & pseudo-observations

Here is the idea that makes the whole subject click. Sklar’s theorem (1959) says any joint distribution factors cleanly into two parts: the **marginals** — what each market does on its own — and a **copula**, a joint distribution on the unit square that carries all of the togetherness and none of the marginal shape. Separate the dancers from the dance. To see the copula empirically, replace each return by its normalised rank, $u = \frac{\operatorname{rank}(x)}{n + 1}$ rank ( x ) ​ — the probability integral transform done with the empirical CDF. Ranks are uniform by construction, so the marginals are washed out entirely — any structure that survives in the picture below is pure dependence.

If the two markets were independent this square would be filled uniformly. Instead the mass drains toward the diagonal — and, critically, **piles up in the corners**. The lower-left corner is the object of study for the rest of this article: weeks in which both markets were simultaneously in their worst tail.

In [ ]:
def pseudo_obs(x):
    return stats.rankdata(x) / (len(x) + 1)

U = pd.DataFrame({c: pseudo_obs(ret[c].values) for c in ret.columns},
                 index=ret.index)

# sanity check: this is just the ECDF rescaled by n/(n+1)
ecdf = ECDF(ret["SPX"].values)
assert np.allclose(U["SPX"], ecdf(ret["SPX"].values) * len(ret) / (len(ret) + 1))

g = sns.jointplot(x=U["SPX"], y=U["FTSE"], kind="scatter", s=10, alpha=0.35,
                  marginal_kws=dict(bins=25))
g.set_axis_labels("u = F(SPX weekly return)", "v = F(FTSE weekly return)")
g.figure.suptitle("Pseudo-observations — note the (0,0) corner", y=1.02);

## 3. The Gaussian copula — elegant, and wrong in the corner

The Gaussian copula is what you implicitly assume whenever you summarise joint behaviour with a correlation matrix alone. Fitting it is one line: push the pseudo-observations through the standard normal quantile function (“normal scores”) and take their correlation — for SPX–FTSE that gives $\rho = 0.74$ . It is analytically convenient, it scales to any dimension, and it has one fatal property: **zero tail dependence**. Ask it “given one market is having a 1-in-q disaster, how often is the other one too?” and as q deepens its answer marches to zero — for any ρ < 1, no matter how high. In the limit, joint crashes are not just rare. They are assumed away.

You can watch the assumption fail at finite depth. At the 10% level the fitted Gaussian copula implies a joint-crash ratio of 50.8% for SPX–FTSE; at 5% it has slipped to 43.5% ; at 1% it is down to 31.1% , on its way to zero. The empirical series goes the other way: 55.3% at 10%, 55.3% at 5%, 61.4% at 1%. The deeper you look into the tail, the more the data diverges from the model — in the direction that hurts.

In [ ]:
Z = stats.norm.ppf(U)                      # normal scores
rho_gauss = pd.DataFrame(np.corrcoef(Z, rowvar=False),
                         index=U.columns, columns=U.columns)
print("Gaussian-copula correlation (normal scores):")
print(rho_gauss.round(3))

## 4. The t copula & tail dependence

The Student-t copula adds exactly one parameter — the degrees of freedom ν — and that single knob buys tail dependence. We set ρ by Kendall’s τ inversion, $\rho = \sin\!\left(\tfrac{\pi\tau}{2}\right)$ ( 2 πτ ​ ) , which is exact for elliptical copulas, and profile the exact copula log-likelihood over a ν grid from 2 to 30 on the SPX–FTSE pair:

The likelihood picks $\nu = 3.5$ — heavy joint tails — and prefers the t copula decisively: log-likelihood 577.9 against the Gaussian’s 526.8 on the same pseudo-observations, one extra parameter. Unlike the Gaussian, the t copula’s tail dependence does not vanish: it converges to the closed-form λ above, which at ν = 3.5 gives 45.9% for SPX–FTSE, 30.9% for SPX–NKY and 29.2% for FTSE–NKY.

Read the teal-vs-graphite gap per pair: the Gaussian copula, fitted to the **same data** with the **same correlation**, undershoots the observed 5% joint-crash ratio for every pair — and remember its bar keeps falling as q shrinks while the empirical one rises. The amber bar is the t copula’s limiting λ: a floor that does not decay, sitting close to what the data shows.

In [ ]:
def t_copula_loglik(u, v, rho, df):
    x, y = stats.t.ppf(u, df), stats.t.ppf(v, df)
    det  = 1 - rho**2
    quad = (x**2 - 2*rho*x*y + y**2) / det
    ll = (gammaln((df+2)/2) + gammaln(df/2) - 2*gammaln((df+1)/2)
          - 0.5*np.log(det)
          - (df+2)/2 * np.log1p(quad/df)
          + (df+1)/2 * (np.log1p(x**2/df) + np.log1p(y**2/df)))
    return ll.sum()

def gauss_copula_loglik(u, v, rho):
    x, y = stats.norm.ppf(u), stats.norm.ppf(v)
    det  = 1 - rho**2
    ll = -0.5*np.log(det) - (rho**2*(x**2+y**2) - 2*rho*x*y) / (2*det)
    return ll.sum()

tau  = ret["SPX"].corr(ret["FTSE"], method="kendall")
rho  = np.sin(np.pi * tau / 2)                 # tau inversion
grid = np.arange(2.0, 30.5, 0.5)
ll   = np.array([t_copula_loglik(U["SPX"], U["FTSE"], rho, df) for df in grid])
df_hat = grid[ll.argmax()]

print(f"Kendall tau = {tau:.3f}  ->  rho = {rho:.3f}")
print(f"fitted df   = {df_hat:.1f}")
print(f"log-lik: t copula {ll.max():.1f}  vs  Gaussian "
      f"{gauss_copula_loglik(U['SPX'], U['FTSE'], rho_gauss.loc['SPX','FTSE']):.1f}")

plt.plot(grid, ll); plt.axvline(df_hat, ls="--", c="k")
plt.xlabel("degrees of freedom"); plt.ylabel("log-likelihood")
plt.title("Profile likelihood over df — SPX-FTSE t copula");
plt.show()

In [ ]:
def bvn_cdf(a, rho, n=8001):
    z = np.linspace(-9.0, a, n)
    inner = stats.norm.cdf((a - rho*z) / np.sqrt(1 - rho**2))
    return np.trapezoid(inner * stats.norm.pdf(z), z)

def t_lambda(rho, df):
    return 2 * stats.t.cdf(-np.sqrt((df+1)*(1-rho)/(1+rho)), df=df+1)

pairs = [("SPX", "FTSE"), ("SPX", "NKY"), ("FTSE", "NKY")]
print(f"{'pair':<10}{'q':>5}{'empirical':>11}{'Gaussian':>10}{'t-copula':>10}")
for a, b in pairs:
    rho_g = rho_gauss.loc[a, b]
    rho_p = np.sin(np.pi * ret[a].corr(ret[b], method='kendall') / 2)
    lam   = t_lambda(rho_p, df_hat)
    for q in (0.01, 0.05, 0.10):
        emp = ((U[a] <= q) & (U[b] <= q)).mean() / q
        gau = bvn_cdf(stats.norm.ppf(q), rho_g) / q
        print(f"{a}-{b:<6}{q:>5.0%}{emp:>11.3f}{gau:>10.3f}{lam:>10.3f}")

## 5. Conditional crash probabilities

The same mathematics, phrased the way a risk committee asks it: given that one market has a worst-decile week, what is the probability the other one does too? Under independence the answer would be 10%. The data answers between 37.7% and 55.4% :

A worst-decile S&P week drags the FTSE into its own worst decile more than half the time — 72 of 130 of the 130 conditioning weeks — and even the geographically and temporally distant Nikkei follows 42.3% of the time. Note the ordering in every row: empirical ≥ t copula λ > Gaussian-at-depth. The single-parameter fix gets you most, not all, of the way — the residual gap is the asymmetry between crash and boom corners that symmetric elliptical copulas cannot express.

In [ ]:
thr = 0.10
for a, b in pairs:
    mask = U[a] <= thr
    p = (mask & (U[b] <= thr)).sum() / mask.sum()
    print(f"P({b} worst decile | {a} worst decile) = {p:.1%}   (independence: 10.0%)")

## 6. The practitioner take

For multi-asset stress testing, the workflow this tutorial rehearses is the one that survives contact with a crisis:

- **Model marginals separately** — fat tails, volatility clustering, whatever each series needs — then choose the dependence structure as a deliberate act, not as a side effect of writing down a correlation matrix.
- **Check the candidate copula against the data** — compare it to the empirical joint-crash ratios at several depths before trusting it.
- **When in doubt between Gaussian and t, pay the one parameter**— the cheapest tail insurance in the toolbox: here it turned “joint crashes become impossible” into λ ≈ 46% for the closest pair.
- **Know where this road ends** — the t copula is elliptical, so its lower-tail dependence comes bundled with an identical upper tail: joint booms priced as generously as joint crashes. When that asymmetry matters, the next tools up are the asymmetric Archimedean families (Clayton glues lower tails only) and vine constructions, which assemble a high-dimensional copula from freely chosen pairs.

- Sklar, A. (1959). Fonctions de répartition à n dimensions et leurs marges. Publications de l'Institut de Statistique de l'Université de Paris, 8, 229–231.
- Embrechts, P., McNeil, A. & Straumann, D. (2002). Correlation and dependence in risk management: properties and pitfalls. In Risk Management: Value at Risk and Beyond, Cambridge University Press.
- Demarta, S. & McNeil, A. J. (2005). The t copula and related copulas. International Statistical Review, 73(1), 111–129.
- Aas, K., Czado, C., Frigessi, A. & Bakken, H. (2009). Pair-copula constructions of multiple dependence. Insurance: Mathematics and Economics, 44(2), 182–198.
- Li, D. X. (2000). On default correlation: a copula function approach. Journal of Fixed Income, 9(4), 43–54.
- Companion notebook: `copulas-tail-dependence.ipynb` — reproduces every figure from raw data (seed 11 ).